In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer

# Load dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(df.head())

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0             

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector

# Model
model = LogisticRegression(max_iter=5000)

# SFS
sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='forward')
sfs.fit(X_train, y_train)

# Selected features
sfs_features = X.columns[sfs.get_support()]

print("Top features (SFS):")
print(sfs_features)

Top features (SFS):
Index(['mean radius', 'worst radius', 'worst texture', 'worst smoothness',
       'worst concavity'],
      dtype='object')


In [ ]:
# SBE (backward)
sbe = SequentialFeatureSelector(model, n_features_to_select=5, direction='backward')
sbe.fit(X_train, y_train)

# Selected features
sbe_features = X.columns[sbe.get_support()]

print("Top features (SBE):")
print(sbe_features)

Top features (SBE):
Index(['worst texture', 'worst area', 'worst smoothness', 'worst concavity',
       'worst fractal dimension'],
      dtype='object')


In [ ]:
from sklearn.metrics import accuracy_score

# Convert back to DataFrame for column selection
X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_test_df = pd.DataFrame(X_test, columns=X.columns)

# SFS model
model.fit(X_train_df[sfs_features], y_train)
y_pred_sfs = model.predict(X_test_df[sfs_features])
acc_sfs = accuracy_score(y_test, y_pred_sfs)

# SBE model
model.fit(X_train_df[sbe_features], y_train)
y_pred_sbe = model.predict(X_test_df[sbe_features])
acc_sbe = accuracy_score(y_test, y_pred_sbe)

print("\nAccuracy using SFS:", acc_sfs)
print("Accuracy using SBE:", acc_sbe)


Accuracy using SFS: 0.9736842105263158
Accuracy using SBE: 0.9736842105263158


In [ ]:
print("\n=== FINAL COMPARISON ===")
print("SFS Selected Features:", list(sfs_features))
print("SBE Selected Features:", list(sbe_features))

if acc_sfs > acc_sbe:
    print("\nSFS performs better")
elif acc_sbe > acc_sfs:
    print("\nSBE performs better")
else:
    print("\nBoth perform equally")


=== FINAL COMPARISON ===
SFS Selected Features: ['mean radius', 'worst radius', 'worst texture', 'worst smoothness', 'worst concavity']
SBE Selected Features: ['worst texture', 'worst area', 'worst smoothness', 'worst concavity', 'worst fractal dimension']

Both perform equally
